In [ ]:
import pandas as pd
import numpy as np
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
import re
nltk.download('stopwords')
from nltk.corpus import stopwords

df = pd.read_csv("/mnt/data/indeed_results.csv")


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stopwords.words('english')]
    return ' '.join(words)

df['Cleaned_Summary'] = df['summary'].apply(clean_text)

vectorizer = TfidfVectorizer(max_features=2000)
X = vectorizer.fit_transform(df['Cleaned_Summary'])

y_title = df['title']
X_train, X_test, y_train, y_test = train_test_split(X, y_title, test_size=0.2, random_state=42)
title_model = LogisticRegression(max_iter=1000)
title_model.fit(X_train, y_train)
y_pred_title = title_model.predict(X_test)
print("Job Title Prediction Accuracy:", accuracy_score(y_test, y_pred_title))

y_salary = df['salary_avg']
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y_salary, test_size=0.2, random_state=42)
salary_model = LinearRegression()
salary_model.fit(X_train_s, y_train_s)
y_pred_salary = salary_model.predict(X_test_s)
print("Mean Absolute Error (Salary Prediction):", mean_absolute_error(y_test_s, y_pred_salary))

test_job = "We are looking for a data analyst with SQL, Python and visualization experience."
cleaned_test = clean_text(test_job)
vector_test = vectorizer.transform([cleaned_test])
predicted_title = title_model.predict(vector_test)[0]
predicted_salary = salary_model.predict(vector_test)[0]

print("Input Job Description:", test_job)
print("Predicted Job Title:", predicted_title)
print("Predicted Salary: $", round(predicted_salary, 2))
